In [2]:
from pathlib import Path
import pandas as pd


# ============================================================
# 1. CONFIGURAÇÃO
# ============================================================

PROJECT_ROOT = Path(".")

CSV_PATH = PROJECT_ROOT / "data" / "selected_30k.csv"
IMAGE_ROOT = PROJECT_ROOT / "data" / "BigEarthNet-S2"

EXPECTED_BANDS = [
    "B01",
    "B02",
    "B03",
    "B04",
    "B05",
    "B06",
    "B07",
    "B08",
    "B09",
    "B11",
    "B12",
    "B8A",
]

EXPECTED_N_PATCHES = 30_000


# ============================================================
# 2. CARREGAMENTO
# ============================================================

print("=" * 70)
print("AUDITORIA DO BIGEARTHNET — selected_30k")
print("=" * 70)

if not CSV_PATH.exists():
    raise FileNotFoundError(
        f"CSV não encontrado em: {CSV_PATH.resolve()}"
    )

if not IMAGE_ROOT.exists():
    raise FileNotFoundError(
        f"Diretório de imagens não encontrado em: {IMAGE_ROOT.resolve()}"
    )

df = pd.read_csv(CSV_PATH)

print("\n[1] CSV carregado")
print(f"Linhas: {len(df):,}")
print(f"Colunas: {len(df.columns)}")


# ============================================================
# 3. ESTRUTURA DO CSV
# ============================================================

print("\n" + "=" * 70)
print("2. ESTRUTURA DO CSV")
print("=" * 70)

print("\nColunas:")
for i, column in enumerate(df.columns, start=1):
    print(f"{i:2d}. {column}")


print("\nTipos:")
print(df.dtypes)


print("\nPrimeiras linhas:")
print(df.head())


# ============================================================
# 4. TAMANHO ESPERADO
# ============================================================

print("\n" + "=" * 70)
print("3. TAMANHO DO DATASET")
print("=" * 70)

print(f"Patches encontrados : {len(df):,}")
print(f"Patches esperados   : {EXPECTED_N_PATCHES:,}")

if len(df) == EXPECTED_N_PATCHES:
    print("✓ Quantidade correta de patches.")
else:
    print("⚠ Quantidade diferente da esperada.")


# ============================================================
# 5. VALORES NULOS
# ============================================================

print("\n" + "=" * 70)
print("4. VALORES NULOS")
print("=" * 70)

null_counts = df.isna().sum()

null_counts = null_counts[null_counts > 0]

if len(null_counts) == 0:
    print("✓ Nenhum valor nulo encontrado.")
else:
    print(null_counts.sort_values(ascending=False))


# ============================================================
# 6. DUPLICATAS
# ============================================================

print("\n" + "=" * 70)
print("5. DUPLICATAS")
print("=" * 70)

duplicated_rows = df.duplicated().sum()

print(f"Linhas completamente duplicadas: {duplicated_rows:,}")


# ============================================================
# 7. IDENTIFICAÇÃO DE POSSÍVEIS COLUNAS IMPORTANTES
# ============================================================

print("\n" + "=" * 70)
print("6. POSSÍVEIS COLUNAS IMPORTANTES")
print("=" * 70)

for column in df.columns:

    name = column.lower()

    if any(term in name for term in ["patch", "id"]):
        print(f"[ID]       {column}")

    if any(term in name for term in ["tile", "scene"]):
        print(f"[TILE]     {column}")

    if any(term in name for term in ["text", "caption", "description"]):
        print(f"[TEXT]     {column}")

    if any(term in name for term in ["label", "class", "land", "lc"]):
        print(f"[LABEL]    {column}")


# ============================================================
# 8. DISTRIBUIÇÃO DE VALORES ÚNICOS
# ============================================================

print("\n" + "=" * 70)
print("7. CARDINALIDADE DAS COLUNAS")
print("=" * 70)

cardinality = pd.DataFrame({
    "n_unique": df.nunique(dropna=False),
    "n_null": df.isna().sum(),
})

cardinality["unique_ratio"] = (
    cardinality["n_unique"] / len(df)
)

print(
    cardinality
    .sort_values("n_unique")
    .to_string()
)


# ============================================================
# 9. IDENTIFICAR COLUNA DO PATCH
# ============================================================

patch_candidates = [
    column
    for column in df.columns
    if "patch" in column.lower()
]

print("\n" + "=" * 70)
print("8. IDENTIFICAÇÃO DOS PATCHES")
print("=" * 70)

if patch_candidates:
    print("Possíveis colunas de patch:")
    for column in patch_candidates:
        print(f" - {column}")

        print(
            df[column]
            .dropna()
            .astype(str)
            .head(10)
            .tolist()
        )
else:
    print("⚠ Nenhuma coluna contendo 'patch' foi encontrada.")


# ============================================================
# 10. DUPLICATAS DO PATCH
# ============================================================

if patch_candidates:

    patch_column = patch_candidates[0]

    duplicated_patches = (
        df[patch_column]
        .duplicated()
        .sum()
    )

    print("\n" + "=" * 70)
    print("9. DUPLICATAS DE PATCH")
    print("=" * 70)

    print(f"Coluna utilizada: {patch_column}")
    print(f"Patches duplicados: {duplicated_patches:,}")


# ============================================================
# 11. IDENTIFICAÇÃO DO TILE
# ============================================================

tile_candidates = [
    column
    for column in df.columns
    if "tile" in column.lower()
    or "scene" in column.lower()
]

print("\n" + "=" * 70)
print("10. TILES")
print("=" * 70)

if tile_candidates:

    for column in tile_candidates:

        print(f"\nColuna: {column}")

        n_tiles = df[column].nunique()

        print(f"Número de valores únicos: {n_tiles:,}")

        print("\nPatches por tile:")

        print(
            df[column]
            .value_counts()
            .describe()
        )

        print("\nTop 10 tiles:")

        print(
            df[column]
            .value_counts()
            .head(10)
        )

else:
    print("⚠ Nenhuma coluna de tile encontrada.")


# ============================================================
# 12. TEXTOS
# ============================================================

text_candidates = [
    column
    for column in df.columns
    if any(
        term in column.lower()
        for term in [
            "text",
            "caption",
            "description",
            "sentence",
            "phrase"
        ]
    )
]

print("\n" + "=" * 70)
print("11. DESCRIÇÕES TEXTUAIS")
print("=" * 70)

if text_candidates:

    for column in text_candidates:

        print(f"\nColuna: {column}")

        series = df[column].dropna().astype(str)

        print(f"Textos não nulos: {len(series):,}")
        print(f"Textos únicos: {series.nunique():,}")

        lengths = series.str.split().str.len()

        print("\nEstatísticas do tamanho:")
        print(lengths.describe())

        print("\nExemplos:")

        for text in series.head(5):
            print(f" - {text}")

else:
    print("⚠ Nenhuma coluna textual identificada.")


# ============================================================
# 13. VERIFICAÇÃO DA ESTRUTURA DAS IMAGENS
# ============================================================

print("\n" + "=" * 70)
print("12. ESTRUTURA DAS IMAGENS")
print("=" * 70)

tiles = [
    path
    for path in IMAGE_ROOT.iterdir()
    if path.is_dir()
]

print(f"Diretórios encontrados em BigEarthNet-S2: {len(tiles):,}")


# ============================================================
# 14. CONTAGEM DE PATCHES NO DISCO
# ============================================================

print("\nContando diretórios de patches...")

patch_directories = []

for tile_dir in tiles:

    for patch_dir in tile_dir.iterdir():

        if patch_dir.is_dir():
            patch_directories.append(patch_dir)


print(
    f"Diretórios de patches encontrados: "
    f"{len(patch_directories):,}"
)


# ============================================================
# 15. VERIFICAÇÃO DAS BANDAS
# ============================================================

print("\n" + "=" * 70)
print("13. VERIFICAÇÃO DAS BANDAS")
print("=" * 70)

band_counts = {}

for patch_dir in patch_directories:

    tif_files = list(patch_dir.glob("*.tif"))

    bands_found = set()

    for tif in tif_files:

        name = tif.stem

        for band in EXPECTED_BANDS:

            if name.endswith(f"_{band}"):
                bands_found.add(band)
                break

    band_counts[len(bands_found)] = (
        band_counts.get(len(bands_found), 0) + 1
    )


print("\nNúmero de bandas por patch:")

for n_bands, count in sorted(band_counts.items()):
    print(
        f"{n_bands:2d} bandas → "
        f"{count:,} patches"
    )


# ============================================================
# 16. PATCHES COM BANDAS AUSENTES
# ============================================================

print("\nVerificando patches incompletos...")

incomplete_patches = []

for patch_dir in patch_directories:

    existing = {
        tif.stem.split("_")[-1]
        for tif in patch_dir.glob("*.tif")
    }

    missing = set(EXPECTED_BANDS) - existing

    if missing:

        incomplete_patches.append({
            "patch_id": patch_dir.name,
            "missing_bands": sorted(missing),
        })


print(
    f"Patches com bandas ausentes: "
    f"{len(incomplete_patches):,}"
)

if incomplete_patches:

    print("\nPrimeiros exemplos:")

    for item in incomplete_patches[:10]:
        print(item)


# ============================================================
# 17. RESULTADO FINAL
# ============================================================

print("\n" + "=" * 70)
print("AUDITORIA CONCLUÍDA")
print("=" * 70)

print(
    f"""
CSV:
    {len(df):,} linhas

Imagens:
    {len(tiles):,} tiles
    {len(patch_directories):,} patches

Bandas esperadas:
    {len(EXPECTED_BANDS)}

Patches incompletos:
    {len(incomplete_patches):,}

Linhas duplicadas:
    {duplicated_rows:,}
"""
)


AUDITORIA DO BIGEARTHNET — selected_30k

[1] CSV carregado
Linhas: 30,000
Colunas: 9

2. ESTRUTURA DO CSV

Colunas:
 1. patch_id
 2. input
 3. output
 4. split
 5. latitude
 6. longitude
 7. country
 8. season
 9. climate_zone

Tipos:
patch_id         object
input            object
output           object
split            object
latitude        float64
longitude       float64
country          object
season           object
climate_zone     object
dtype: object

Primeiras linhas:
                                            patch_id  \
0  S2A_MSIL2A_20180225T114351_N9999_R123_T29UPU_0...   
1  S2B_MSIL2A_20171015T104009_N9999_R008_T31UGR_1...   
2  S2A_MSIL2A_20171002T094031_N9999_R036_T34TCR_7...   
3  S2A_MSIL2A_20171101T094131_N9999_R036_T35VNK_6...   
4  S2B_MSIL2A_20180515T112109_N9999_R037_T29SNC_0...   

                                               input  \
0  Explain the content of the image, highlighting...   
1  Explain what can be seen in this satellite ima...   
2  Explain 

In [7]:
from pathlib import Path
from collections import Counter, defaultdict
import pandas as pd
import rasterio
import numpy as np

# ============================================================
# CONFIGURAÇÃO
# ============================================================

PROJECT_ROOT = Path(".")
IMAGE_ROOT = PROJECT_ROOT / "data" / "BigEarthNet-S2"
CSV_PATH = PROJECT_ROOT / "data" / "selected_30k.csv"

BANDS = [
    "B01", "B02", "B03", "B04",
    "B05", "B06", "B07", "B08",
    "B09", "B11", "B12", "B8A"
]

# ============================================================
# CARREGAR CSV
# ============================================================

df = pd.read_csv(CSV_PATH)

print("=" * 70)
print("AUDITORIA DOS TIFFs")
print("=" * 70)

print(f"\nPatches no CSV: {len(df):,}")
print(f"IMAGE_ROOT: {IMAGE_ROOT.resolve()}")
print(f"IMAGE_ROOT existe: {IMAGE_ROOT.exists()}")

# ============================================================
# FUNÇÃO PARA OBTER O TILE
# ============================================================

def get_tile_from_patch_id(patch_id):
    return patch_id.rsplit("_", 2)[0]


# ============================================================
# 1. VERIFICAR PATCHES
# ============================================================

missing_patches = []
existing_patches = []

for patch_id in df["patch_id"]:

    tile = get_tile_from_patch_id(patch_id)

    patch_dir = IMAGE_ROOT / tile / patch_id

    if patch_dir.is_dir():
        existing_patches.append(patch_id)
    else:
        missing_patches.append(patch_id)

print("\n" + "=" * 70)
print("1. EXISTÊNCIA DOS PATCHES")
print("=" * 70)

print(f"Patches encontrados: {len(existing_patches):,}")
print(f"Patches ausentes:   {len(missing_patches):,}")

if missing_patches:
    print("\nPrimeiros patches ausentes:")
    for patch_id in missing_patches[:20]:
        tile = get_tile_from_patch_id(patch_id)
        print(f"  {IMAGE_ROOT / tile / patch_id}")


# ============================================================
# 2. VERIFICAR BANDAS
# ============================================================

band_counts = Counter()
band_missing = defaultdict(list)
read_errors = []

all_band_records = []

for patch_id in existing_patches:

    tile = get_tile_from_patch_id(patch_id)
    patch_dir = IMAGE_ROOT / tile / patch_id

    for band in BANDS:

        tif_path = patch_dir / f"{patch_id}_{band}.tif"

        if not tif_path.exists():
            band_missing[band].append(patch_id)
            continue

        band_counts[band] += 1

        try:
            with rasterio.open(tif_path) as src:

                all_band_records.append({
                    "patch_id": patch_id,
                    "band": band,
                    "shape": src.shape,
                    "width": src.width,
                    "height": src.height,
                    "resolution_x": src.res[0],
                    "resolution_y": src.res[1],
                    "dtype": src.dtypes[0],
                    "nodata": src.nodata,
                    "crs": str(src.crs),
                    "transform": src.transform
                })

        except Exception as e:
            read_errors.append({
                "patch_id": patch_id,
                "band": band,
                "error": str(e)
            })


# ============================================================
# 3. RESUMO DAS BANDAS
# ============================================================

print("\n" + "=" * 70)
print("2. BANDAS")
print("=" * 70)

for band in BANDS:

    total = band_counts[band]
    missing = len(band_missing[band])

    print(
        f"{band}: "
        f"{total:,} encontrados | "
        f"{missing:,} ausentes"
    )


# ============================================================
# 4. SHAPE
# ============================================================

print("\n" + "=" * 70)
print("3. SHAPE DAS BANDAS")
print("=" * 70)

records_df = pd.DataFrame(all_band_records)

if not records_df.empty:

    for band in BANDS:

        subset = records_df[records_df["band"] == band]

        print(f"\n{band}:")
        print(subset["shape"].value_counts())


# ============================================================
# 5. RESOLUÇÃO
# ============================================================

print("\n" + "=" * 70)
print("4. RESOLUÇÃO ESPACIAL")
print("=" * 70)

if not records_df.empty:

    for band in BANDS:

        subset = records_df[records_df["band"] == band]

        print(f"\n{band}:")
        print(
            subset[
                ["resolution_x", "resolution_y"]
            ].value_counts()
        )


# ============================================================
# 6. DTYPE
# ============================================================

print("\n" + "=" * 70)
print("5. DTYPE")
print("=" * 70)

if not records_df.empty:

    for band in BANDS:

        subset = records_df[records_df["band"] == band]

        print(f"\n{band}:")
        print(subset["dtype"].value_counts())


# ============================================================
# 7. NODATA
# ============================================================

print("\n" + "=" * 70)
print("6. NODATA")
print("=" * 70)

if not records_df.empty:

    for band in BANDS:

        subset = records_df[records_df["band"] == band]

        nodata_count = subset["nodata"].notna().sum()

        print(
            f"{band}: "
            f"{nodata_count:,}/{len(subset):,} "
            f"arquivos com NoData"
        )


# ============================================================
# 8. ESTATÍSTICAS DE PIXELS
#    AMOSTRA DE ATÉ 100 PATCHES
# ============================================================

print("\n" + "=" * 70)
print("7. ESTATÍSTICAS DOS PIXELS")
print("=" * 70)

sample_patches = existing_patches[:100]

pixel_stats = []

for patch_id in sample_patches:

    tile = get_tile_from_patch_id(patch_id)
    patch_dir = IMAGE_ROOT / tile / patch_id

    for band in BANDS:

        tif_path = patch_dir / f"{patch_id}_{band}.tif"

        if not tif_path.exists():
            continue

        try:
            with rasterio.open(tif_path) as src:

                data = src.read(1).astype(np.float32)

                # Remove NaN/infinito
                data = data[np.isfinite(data)]

                if len(data) == 0:
                    continue

                pixel_stats.append({
                    "band": band,
                    "min": float(data.min()),
                    "max": float(data.max()),
                    "mean": float(data.mean()),
                    "std": float(data.std())
                })

        except Exception as e:
            pass


stats_df = pd.DataFrame(pixel_stats)

if not stats_df.empty:

    print(
        stats_df
        .groupby("band")[["min", "max", "mean", "std"]]
        .agg(["min", "max", "mean", "std"])
        .round(4)
    )


# ============================================================
# 9. ERROS
# ============================================================

print("\n" + "=" * 70)
print("8. ERROS DE LEITURA")
print("=" * 70)

print(f"Erros: {len(read_errors)}")

for error in read_errors[:20]:
    print(error)


# ============================================================
# 10. EXEMPLO DA ESTRUTURA
# ============================================================

print("\n" + "=" * 70)
print("9. EXEMPLO DE ESTRUTURA")
print("=" * 70)

if existing_patches:

    patch_id = existing_patches[0]
    tile = get_tile_from_patch_id(patch_id)
    patch_dir = IMAGE_ROOT / tile / patch_id

    print(f"\nTile:")
    print(f"  {tile}")

    print(f"\nPatch:")
    print(f"  {patch_dir}")

    print("\nArquivos:")

    for p in sorted(patch_dir.iterdir()):
        print(f"  {p.name}")


# ============================================================
# RESUMO
# ============================================================

print("\n" + "=" * 70)
print("RESUMO FINAL")
print("=" * 70)

print(f"""
Patches no CSV:          {len(df):,}
Patches encontrados:     {len(existing_patches):,}
Patches ausentes:        {len(missing_patches):,}
Arquivos TIFF analisados:{len(all_band_records):,}
Erros de leitura:        {len(read_errors):,}
""")

print("=" * 70)
print("AUDITORIA CONCLUÍDA")
print("=" * 70)

AUDITORIA DOS TIFFs

Patches no CSV: 30,000
IMAGE_ROOT: /home/ettore/Documentos/PUC/Redes Neurais Proj 5/data/BigEarthNet-S2
IMAGE_ROOT existe: True

1. EXISTÊNCIA DOS PATCHES
Patches encontrados: 30,000
Patches ausentes:   0

2. BANDAS
B01: 30,000 encontrados | 0 ausentes
B02: 30,000 encontrados | 0 ausentes
B03: 30,000 encontrados | 0 ausentes
B04: 30,000 encontrados | 0 ausentes
B05: 30,000 encontrados | 0 ausentes
B06: 30,000 encontrados | 0 ausentes
B07: 30,000 encontrados | 0 ausentes
B08: 30,000 encontrados | 0 ausentes
B09: 30,000 encontrados | 0 ausentes
B11: 30,000 encontrados | 0 ausentes
B12: 30,000 encontrados | 0 ausentes
B8A: 30,000 encontrados | 0 ausentes

3. SHAPE DAS BANDAS

B01:
shape
(20, 20)    30000
Name: count, dtype: int64

B02:
shape
(120, 120)    30000
Name: count, dtype: int64

B03:
shape
(120, 120)    30000
Name: count, dtype: int64

B04:
shape
(120, 120)    30000
Name: count, dtype: int64

B05:
shape
(60, 60)    30000
Name: count, dtype: int64

B06:
shape


In [8]:
from pathlib import Path
from collections import defaultdict, Counter
import pandas as pd
import rasterio
import numpy as np

# ============================================================
# CONFIGURAÇÃO
# ============================================================

PROJECT_ROOT = Path(".")
IMAGE_ROOT = PROJECT_ROOT / "data" / "BigEarthNet-S2"
CSV_PATH = PROJECT_ROOT / "data" / "selected_30k.csv"

BANDS = [
    "B01", "B02", "B03", "B04",
    "B05", "B06", "B07", "B08",
    "B09", "B11", "B12", "B8A"
]

df = pd.read_csv(CSV_PATH)


def get_tile_from_patch_id(patch_id):
    return patch_id.rsplit("_", 2)[0]


# ============================================================
# 1. DESCOBRIR VALORES NODATA DECLARADOS
# ============================================================

nodata_values = defaultdict(Counter)

print("=" * 70)
print("1. VALORES NODATA DECLARADOS")
print("=" * 70)

for patch_id in df["patch_id"]:

    tile = get_tile_from_patch_id(patch_id)
    patch_dir = IMAGE_ROOT / tile / patch_id

    for band in BANDS:

        tif_path = patch_dir / f"{patch_id}_{band}.tif"

        with rasterio.open(tif_path) as src:
            nodata = src.nodata

        nodata_values[band][nodata] += 1


for band in BANDS:

    print(f"\n{band}:")

    for value, count in nodata_values[band].items():

        print(
            f"  NoData = {value} "
            f"→ {count:,} arquivos"
        )


# ============================================================
# 2. CONTAR PIXELS NODATA
# ============================================================

print("\n" + "=" * 70)
print("2. QUANTIDADE DE PIXELS NODATA")
print("=" * 70)

results = []

for band in BANDS:

    total_pixels = 0
    nodata_pixels = 0
    valid_pixels = 0

    # Valor NoData esperado mais frequente
    expected_nodata = nodata_values[band].most_common(1)[0][0]

    for patch_id in df["patch_id"]:

        tile = get_tile_from_patch_id(patch_id)
        patch_dir = IMAGE_ROOT / tile / patch_id

        tif_path = patch_dir / f"{patch_id}_{band}.tif"

        with rasterio.open(tif_path) as src:

            data = src.read(1)

            total = data.size

            if expected_nodata is None:

                # Caso não exista NoData declarado
                mask_nodata = np.zeros(
                    data.shape,
                    dtype=bool
                )

            else:

                mask_nodata = data == expected_nodata

            n_nodata = int(mask_nodata.sum())
            n_valid = total - n_nodata

            total_pixels += total
            nodata_pixels += n_nodata
            valid_pixels += n_valid

    percentage = (
        100 * nodata_pixels / total_pixels
        if total_pixels > 0
        else 0
    )

    results.append({
        "band": band,
        "nodata_value": expected_nodata,
        "total_pixels": total_pixels,
        "nodata_pixels": nodata_pixels,
        "valid_pixels": valid_pixels,
        "nodata_percent": percentage
    })


nodata_df = pd.DataFrame(results)

print(
    nodata_df.to_string(
        index=False,
        formatters={
            "nodata_percent": "{:.4f}%".format
        }
    )
)


# ============================================================
# 3. DISTRIBUIÇÃO DO NODATA POR PATCH
# ============================================================

print("\n" + "=" * 70)
print("3. DISTRIBUIÇÃO DE NODATA POR PATCH")
print("=" * 70)

for band in BANDS:

    expected_nodata = nodata_values[band].most_common(1)[0][0]

    percentages = []

    for patch_id in df["patch_id"]:

        tile = get_tile_from_patch_id(patch_id)
        patch_dir = IMAGE_ROOT / tile / patch_id

        tif_path = patch_dir / f"{patch_id}_{band}.tif"

        with rasterio.open(tif_path) as src:

            data = src.read(1)

            if expected_nodata is None:
                n_nodata = 0
            else:
                n_nodata = np.sum(data == expected_nodata)

            percentages.append(
                100 * n_nodata / data.size
            )

    percentages = np.array(percentages)

    print(f"\n{band}:")
    print(f"  Mínimo:  {percentages.min():.4f}%")
    print(f"  Mediana: {np.median(percentages):.4f}%")
    print(f"  Média:   {percentages.mean():.4f}%")
    print(f"  Máximo:  {percentages.max():.4f}%")

1. VALORES NODATA DECLARADOS

B01:
  NoData = 0.0 → 30,000 arquivos

B02:
  NoData = 0.0 → 30,000 arquivos

B03:
  NoData = 0.0 → 30,000 arquivos

B04:
  NoData = 0.0 → 30,000 arquivos

B05:
  NoData = 0.0 → 30,000 arquivos

B06:
  NoData = 0.0 → 30,000 arquivos

B07:
  NoData = 0.0 → 30,000 arquivos

B08:
  NoData = 0.0 → 30,000 arquivos

B09:
  NoData = 0.0 → 30,000 arquivos

B11:
  NoData = 0.0 → 30,000 arquivos

B12:
  NoData = 0.0 → 30,000 arquivos

B8A:
  NoData = 0.0 → 30,000 arquivos

2. QUANTIDADE DE PIXELS NODATA
band  nodata_value  total_pixels  nodata_pixels  valid_pixels nodata_percent
 B01           0.0      12000000              0      12000000        0.0000%
 B02           0.0     432000000              0     432000000        0.0000%
 B03           0.0     432000000              0     432000000        0.0000%
 B04           0.0     432000000              0     432000000        0.0000%
 B05           0.0     108000000              0     108000000        0.0000%
 B06     